# Extended Data Figure 2 — Dependence of credible sets on GWAS sample size

Average number of credible sets per study as a function of log₁₀(N_samples),
shown separately for disease GWAS (blue) and measurement GWAS (green).
Studies are binned into 5 equal-frequency (quantile) bins by sample size.

**Source notebook:** `chapters/02-analysis/01-descriptions-numbers/02_gene_stauration_plots.ipynb`
**Data:** `data/intermediate_files/lead_variant_effect/`, `data/intermediate_files/qualifying_credible_sets/`,
`data/25.06/output/study/`


## Setup


In [ ]:
from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from pyspark.sql import functions as f

In [ ]:
session = Session(extended_spark_conf={"spark.driver.memory": "40G"})

## Paths


In [ ]:
from manuscript_methods import paper

path_to_release_folder = str(paper.ROOT / "data" / "25.06") + "/"
path_to_intermediate_data_folder = str(paper.DERIVED) + "/"
figure_dir = str(paper.ROOT / "chapters" / "05-figures-supplementary" / "extended_data")

qualifying_disease_cs_path = path_to_intermediate_data_folder + "qualifying_credible_sets"
qualifying_measurement_cs_path = path_to_intermediate_data_folder + "qualifying_measurement_credible_sets"
lead_variant_effect_path = path_to_intermediate_data_folder + "lead_variant_effect"
study_index_path = path_to_release_folder + "output/study"

## Load data

`sl_eff` contains one row per (studyLocusId, variantId) – i.e. per credible-set lead variant.
Grouping by studyId gives the number of credible sets per study.
We enrich it with `nSamples` and `year` from the study index.


In [ ]:
# Load study index and add year from publication date
si = StudyIndex.from_parquet(session, study_index_path)
si_with_year = (
    si.df.withColumn(
        "publicationDate",
        f.when(f.col("projectId") == "FINNGEN_R12", f.lit("2024-11-04")).otherwise(f.col("publicationDate")),
    )
    .withColumn("year", f.col("publicationDate").substr(1, 4).cast("int"))
    .select("studyId", "nSamples", "year")
)

In [ ]:
# Load lead variant effect data and join with study metadata
sl_eff = session.spark.read.parquet(lead_variant_effect_path)
sl_eff = sl_eff.join(si_with_year, on="studyId", how="inner").cache()
print(f"Lead variant effects: {sl_eff.count():,}")

## Filter to qualifying disease and measurement credible sets


In [ ]:
# Qualifying disease credible sets
qd_cs = session.spark.read.parquet(qualifying_disease_cs_path).select("studyLocusId").cache()
print(f"Qualifying disease CSs: {qd_cs.count():,}")

# Qualifying measurement credible sets
qm_cs = session.spark.read.parquet(qualifying_measurement_cs_path).select("studyLocusId").cache()
print(f"Qualifying measurement CSs: {qm_cs.count():,}")

In [ ]:
# Filter sl_eff to qualifying CSs and convert to pandas
qd_sl_eff = sl_eff.join(qd_cs, on="studyLocusId", how="inner").toPandas()
qm_sl_eff = sl_eff.join(qm_cs, on="studyLocusId", how="inner").toPandas()
print(f"Disease lead variants: {len(qd_sl_eff):,} | Measurement lead variants: {len(qm_sl_eff):,}")

## Extended Data Figure 2

Bin studies into 5 quantile bins by sample size. For each bin compute the mean ± 95 CI number of credible sets per study.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def plot_cs_vs_sample_size(df, label, color):
    """Plot mean number of credible sets per study (binned by log10 sample size).

    Each row in df is a credible-set lead variant; grouping by studyId gives
    the CS count per study. Studies are binned into 5 equal-frequency quantile
    bins by nSamples, and the mean ± 95 % CI CS count is plotted per bin.
    """
    df_per_study = (
        df.groupby("studyId")
        .agg(
            num_cs=("studyLocusId", "count"),
            nSamples=("nSamples", "first"),
        )
        .reset_index()
    )

    df_per_study["nSamples_bin"] = pd.qcut(df_per_study["nSamples"], q=5, duplicates="drop")
    bin_stats = df_per_study.groupby("nSamples_bin")["num_cs"].agg(["mean", "count", "std"]).reset_index()
    bin_stats["se"] = bin_stats["std"] / np.sqrt(bin_stats["count"])
    bin_stats["ci"] = bin_stats["se"] * 1.96
    bin_stats["bin_mid"] = bin_stats["nSamples_bin"].apply(lambda x: (float(x.left) + float(x.right)) / 2)
    bin_stats["bin_mid_log"] = np.log10(bin_stats["bin_mid"].astype(float).to_numpy())

    plt.errorbar(
        bin_stats["bin_mid_log"],
        bin_stats["mean"],
        yerr=bin_stats["ci"],
        fmt="o-",
        color=color,
        ecolor="gray",
        capsize=3,
        label=label,
    )


fig, ax = plt.subplots(figsize=(6, 4))
plot_cs_vs_sample_size(qd_sl_eff, "disease GWAS", color="royalblue")
plot_cs_vs_sample_size(qm_sl_eff, "measurement GWAS", color="forestgreen")

ax.set_xlabel(r"$\log_{10}(N_\mathrm{samples})$")
ax.set_ylabel("Average number of credible sets per study")
ax.grid(axis="y", color="lightgray", linestyle="--", linewidth=0.7)
ax.legend()
fig.tight_layout()
fig.savefig(f"{figure_dir}/extended_figure_2.pdf", dpi=300, bbox_inches="tight")
plt.show()